# EDA - AndesLink Churn

Análisis exploratorio inicial del dataset sintético de churn.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "churn_sintetico.csv"
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

## Dimensiones y tipos de datos

In [ ]:
print("Filas y columnas:", df.shape)
df.info()

## Valores faltantes

In [ ]:
df.isna().sum().sort_values(ascending=False)

## Distribución de la variable objetivo

In [ ]:
target_counts = df["churn"].value_counts(normalize=True).rename("proportion")
display(target_counts)

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="churn")
plt.title("Distribución de churn")
plt.xlabel("Churn")
plt.ylabel("Cantidad de clientes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "target_distribution.png")
plt.show()

## Variables numéricas

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
df[numeric_cols].describe().T

In [ ]:
for col in ["tenure_months", "monthly_charge", "total_charges", "support_tickets", "late_payments", "avg_monthly_usage_gb", "customer_age"]:
    plt.figure(figsize=(6,4))
    sns.histplot(data=df, x=col, kde=True)
    plt.title(f"Distribución de {col}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"hist_{col}.png")
    plt.show()

## Relación entre variables y churn

In [ ]:
for col in ["contract_type", "payment_method", "internet_service", "region"]:
    plt.figure(figsize=(8,4))
    churn_rate = df.groupby(col)["churn"].mean().sort_values(ascending=False)
    sns.barplot(x=churn_rate.index, y=churn_rate.values)
    plt.title(f"Tasa de churn por {col}")
    plt.ylabel("Tasa de churn")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"churn_rate_{col}.png")
    plt.show()

In [ ]:
plt.figure(figsize=(7,5))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png")
plt.show()

## Conclusiones preliminares

- El dataset no presenta valores faltantes.
- La variable objetivo está desbalanceada de forma moderada: hay más clientes que no abandonan que clientes que abandonan.
- Las variables de contrato, pagos atrasados, soporte, facturación y antigüedad son relevantes para formular hipótesis de churn.
- El siguiente paso es entrenar modelos de clasificación binaria con un pipeline reproducible.